In [1]:
import numpy as np
import matplotlib.pyplot as plt
import astropy 
from astropy.io import fits, ascii
from astropy.table import Table
import astroquery 
import glob
import os
from astroquery.mast import Observations, Mast, MastMissions, MastMissionsClass, Catalogs
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.xmatch import XMatch
from astropy.timeseries import BoxLeastSquares
from astroquery.sdss import SDSS
import fitsio
import pandas as pd

Could not import regions, which is required for some of the functionalities of this module.


In [10]:
files = glob.glob("*")

In [12]:
if('allStar-dr17-synspec_rev1.fits' in files): 
    print("yes")

yes


In [ ]:
def tess_data_download(num_obj = None, order = 'sequential'): 
    if()

In [ ]:
x = None

In [ ]:
boolean = bool

In [ ]:
# from https://exoplanetarchive.ipac.caltech.edu/docs/data.html transiting planets table
transiting_exo = ascii.read('./data/TD_2026.04.27_13.36.06.csv')

In [ ]:
colnames = transiting_exo.colnames
df = pd.DataFrame(np.asarray(transiting_exo))

In [ ]:
transiting_exo_sub = Table(np.asarray(df.drop_duplicates(subset=['pl_name'])), names = colnames)

In [ ]:
row = transiting_exo_sub[0]
ra = row['ra']
dec = row['dec']
search_radius = 0.1
observations = Observations.query_region(f"{ra} {dec}", radius=f"{search_radius} deg")

In [ ]:
obs_wanted = ((observations['dataproduct_type'] == 'timeseries') &
              (observations['obs_collection'] == 'TESS'))

In [ ]:
data_products = Observations.get_product_list(observations[obs_wanted])
products_wanted = Observations.filter_products(data_products, 
                                    productSubGroupDescription=["LC"])
obj_name = row['pl_name']
obj_name=obj_name.replace(" ", "_")
os.system(f"mkdir {obj_name}")
download = Observations.download_products(products_wanted)
for file in download['Local Path']: 
    os.system(f"cp {file} ./{obj_name}")
os.system(f"rm -r mastDownload")

In [ ]:
os.system(f"cp {download['Local Path']} ./{obj_name}")

In [ ]:
row['pl_name']

In [ ]:
associated_star = [False] * len(transiting_exo_sub)
transiting_exo_sub['lumclass'] = ['      '] * len(transiting_exo_sub)
transiting_exo_sub['plx'] = [np.nan]*len(transiting_exo_sub)
transiting_exo_sub['e_plx'] = [np.nan]*len(transiting_exo_sub)
transiting_exo_sub['Teff'] = [np.nan]*len(transiting_exo_sub)
transiting_exo_sub['e_Teff']=[np.nan]*len(transiting_exo_sub)
for i,t in enumerate(transiting_exo_sub): 
    target_name = t['pl_name']
    print(f"Iteration #{i}: {target_name}...")
    search_radius_deg = 0.2
    try:
        catalogTIC = Catalogs.query_object(target_name, radius=search_radius_deg, catalog="TIC")
    except Exception as e: 
        print("Name could not be resolved.")
    where_closest = np.argmin(catalogTIC['dstArcSec'])
    distance = catalogTIC['dstArcSec'][where_closest]
    if(distance < 1):
        print("Match found")
        associated_star[i] = True 
        if(np.ma.is_masked(catalogTIC['lumclass'][where_closest])):
            transiting_exo_sub['lumclass'][i] = 'N/A'
        else: 
            transiting_exo_sub['lumclass'][i] = catalogTIC['lumclass'][where_closest]
        transiting_exo_sub['lumclass'][i] = transiting_exo_sub['lumclass'][i].strip()
        transiting_exo_sub['plx'][i] = catalogTIC['plx'][where_closest]
        transiting_exo_sub['e_plx'][i] = catalogTIC['e_plx'][where_closest]
        transiting_exo_sub['Teff'][i] = catalogTIC['Teff'][where_closest] 
        transiting_exo_sub['e_Teff'][i] = catalogTIC['e_Teff'][where_closest]

In [ ]:
catalogTIC = Catalogs.query_object(target_name, radius=search_radius_deg, catalog="TIC")